# Predicting Undiagnosed Type 2 Diabetes Risk
## A Population Screening Model Trained on South Korean National Health Insurance Data

**Tyson Johnson** · George Mason University, Computational & Data Sciences

---

### Background

Type 2 diabetes mellitus (T2D) is one of the most prevalent and costly chronic diseases
worldwide. In South Korea, the estimated prevalence among adults is approximately 14–16%,
yet a significant proportion of cases remain undiagnosed—individuals who carry the
metabolic burden of the disease without receiving treatment or lifestyle guidance.

Standard diagnostic criteria rely on fasting plasma glucose (FPG ≥ 126 mg/dL) or
glycated haemoglobin (HbA1c ≥ 6.5%). Because these tests require a blood draw ordered
by a clinician, many at-risk individuals are never tested. A non-invasive screening model
that flags high-risk individuals from routinely collected health examination data could
meaningfully improve early detection rates.

### Objective

Train a binary classifier to identify adults who are likely to have undiagnosed T2D,
using the anthropometric, biometric, and laboratory measurements recorded during
routine Korean national health check-ups. The model is designed to operate even when
a full lipid panel is unavailable, as only ~34% of check-up participants receive one.

### Data Source

The dataset is the **Korean National Health Insurance Service (NHIS) General Health
Examination Dataset (2024 cohort)**:
- 1,000,000 randomly sampled adult subscribers
- 33 variables covering demographics, anthropometrics, blood pressure, laboratory results,
  dental findings, and lifestyle factors
- Available from the Korean Open Data Portal:
  https://www.data.go.kr/en/data/15007122/fileData.do

The FPG column was used to construct binary labels; patients with FPG ≥ 126 mg/dL
are labelled as likely undiagnosed T2D.

### Models Evaluated

Three model families are compared across three feature sets of increasing richness:
1. **sklearn MLP** (ANN) — fully-connected neural network with Platt calibration
2. **CatBoost** — gradient-boosted decision trees with native categorical and missing-value handling
3. **PyTorch Tabular** — a custom embedding-based network for mixed numerical/categorical data

The best-performing model is then explained using SHAP (SHapley Additive exPlanations).


## 1. Environment Setup

Install required packages. The `--quiet` flag suppresses verbose pip output on Colab.

In [ ]:
# Uncomment to install in Colab or a fresh environment
# !pip install catboost shap --quiet


## 2. Imports

In [ ]:
from __future__ import annotations

import copy
import math
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import torch
import torch.nn as nn
from catboost import CatBoostClassifier, Pool
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

# Use GPU in Colab if available; fall back to CPU gracefully
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 3. Configuration

All column names, domain constraints, feature lists, and modelling hyperparameters
are defined centrally here so that any change propagates automatically.


In [ ]:
# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
DATA_DIR  = Path("data")
DATA_FILE = DATA_DIR / "health_2024.csv"
DATA_ENCODING = "cp949"
GH_URL = (
    "https://github.com/tjohns94/cds492-3c-project/raw/refs/heads/main/data/health_2024.CSV"
)

# ---------------------------------------------------------------------------
# Raw dataset schema
# ---------------------------------------------------------------------------
COLUMN_NAMES_INIT = [
    "year", "subscriber_id", "city_code",
    "sex_code", "age_group_code", "height",
    "weight", "waist_circumference", "vision_left",
    "vision_right", "hearing_left", "hearing_right",
    "systolic_bp", "diastolic_bp", "fpg",
    "total_cholesterol", "triglycerides", "hdl_cholesterol",
    "ldl_cholesterol", "hemoglobin", "urine_protein",
    "serum_creatinine", "serum_got_ast", "serum_gpt_alt",
    "gamma_gtp", "smoking_status", "alcohol_consumption",
    "oral_exam", "caries_presence", "missing_teeth_presence",
    "tooth_wear_presence", "wisdom_teeth_abnormality", "plaque_presence",
]
COLUMN_NAMES_TO_DROP = [
    "year", "subscriber_id", "missing_teeth_presence",
    "tooth_wear_presence", "wisdom_teeth_abnormality",
]
COLUMN_TYPES = {
    "sex_code":            ["category", "int"],
    "city_code":           ["category", "int"],
    "hearing_left":        ["category", "int"],
    "hearing_right":       ["category", "int"],
    "alcohol_consumption": ["category", "int"],
    "oral_exam":           ["category", "int"],
    "caries_presence":     ["category", "int"],
    "plaque_presence":     ["category", "int"],
    "age_group_code":      ["category", "int"],
    "smoking_status":      ["category", "int"],
    "urine_protein":       ["category", "int"],
    "height":              ["numeric",  "int"],
    "weight":              ["numeric",  "int"],
    "waist_circumference": ["numeric",  "float"],
    "systolic_bp":         ["numeric",  "int"],
    "diastolic_bp":        ["numeric",  "int"],
    "vision_left":         ["numeric",  "float"],
    "vision_right":        ["numeric",  "float"],
    "total_cholesterol":   ["numeric",  "int"],
    "triglycerides":       ["numeric",  "int"],
    "hdl_cholesterol":     ["numeric",  "int"],
    "ldl_cholesterol":     ["numeric",  "int"],
    "hemoglobin":          ["numeric",  "float"],
    "serum_creatinine":    ["numeric",  "float"],
    "serum_got_ast":       ["numeric",  "int"],
    "serum_gpt_alt":       ["numeric",  "int"],
    "gamma_gtp":           ["numeric",  "int"],
    "fpg":                 ["numeric",  "int"],
}
COLUMN_DOMAINS = {
    "sex_code":            {1, 2},
    "city_code":           {11, 26, 27, 28, 29, 30, 31, 36, 41, 42, 43, 44, 45, 46, 47},
    "hearing_left":        {1, 2},
    "hearing_right":       {1, 2},
    "alcohol_consumption": {0, 1},
    "oral_exam":           {0, 1},
    "caries_presence":     {0, 1},
    "plaque_presence":     {0, 1},
    "age_group_code":      set(range(5, 19)),
    "smoking_status":      {1, 2, 3},
    "urine_protein":       set(range(1, 7)),
    "height":              [100, 230],
    "weight":              [20, 250],
    "waist_circumference": [30, 250],
    "systolic_bp":         [60, 300],
    "diastolic_bp":        [30, 200],
    "vision_left":         [0.1, 2.5],
    "vision_right":        [0.1, 2.5],
    "total_cholesterol":   [50, 1000],
    "triglycerides":       [10, 5000],
    "hdl_cholesterol":     [5, 200],
    "ldl_cholesterol":     [5, 700],
    "hemoglobin":          [3, 25],
    "serum_creatinine":    [0.1, 30],
    "serum_got_ast":       [1, 5000],
    "serum_gpt_alt":       [1, 5000],
    "gamma_gtp":           [1, 5000],
    "fpg":                 [30, 1000],
}
FEATURES_MISSING_ALLOWED = {
    "vision_left":     ["blind_left",  1],
    "vision_right":    ["blind_right", 1],
    "caries_presence": ["oral_exam",   0],
    "plaque_presence": ["oral_exam",   0],
}
FEATURES_NUMERIC     = [c for c, (t, _) in COLUMN_TYPES.items() if t == "numeric" and c != "fpg"]
FEATURES_CATEGORICAL = [c for c, (t, _) in COLUMN_TYPES.items() if t == "category"]
FEATURES_ALL         = FEATURES_NUMERIC + FEATURES_CATEGORICAL
TARGET_NUM           = "fpg"
TARGET_CAT           = "has_diabetes"
COLUMNS_ALL          = FEATURES_ALL + [TARGET_NUM]
DIABETES_THRESHOLD   = 126

# ---------------------------------------------------------------------------
# Model feature sets
# ---------------------------------------------------------------------------
CORE_NUMERIC_FEATURES = [
    "age_years_mid", "height", "weight", "waist_circumference",
    "bmi", "waist_to_height", "wwi",
    "systolic_bp", "diastolic_bp", "pulse_pressure", "mean_arterial_pressure",
    "urine_protein",
]
CORE_NOMINAL_FEATURES = ["sex_code", "smoking_status", "alcohol_consumption"]
NONLIPID_LAB_FEATURES = [
    "hemoglobin", "serum_creatinine",
    "log_ast", "log_alt", "log_ggt", "ast_alt_ratio",
]
NONLIPID_MISSING_FLAGS = [
    "missing_hemoglobin", "missing_serum_creatinine",
    "missing_serum_got_ast", "missing_serum_gpt_alt", "missing_gamma_gtp",
]
LIPID_FEATURES = [
    "total_cholesterol", "hdl_cholesterol", "ldl_cholesterol",
    "log_triglycerides", "non_hdl_cholesterol",
    "tg_hdl_ratio", "tc_hdl_ratio", "ldl_hdl_ratio",
    "has_complete_lipid_panel", "n_lipids_available",
]
LIPID_MISSING_FLAGS = [
    "missing_total_cholesterol", "missing_triglycerides",
    "missing_hdl_cholesterol", "missing_ldl_cholesterol",
]
FEATURE_SETS = {
    "core_screening": {
        "features":         CORE_NUMERIC_FEATURES + CORE_NOMINAL_FEATURES,
        "nominal_features": CORE_NOMINAL_FEATURES,
    },
    "core_plus_labs": {
        "features":         CORE_NUMERIC_FEATURES + CORE_NOMINAL_FEATURES
                            + NONLIPID_LAB_FEATURES + NONLIPID_MISSING_FLAGS,
        "nominal_features": CORE_NOMINAL_FEATURES,
    },
    "core_plus_labs_plus_lipids": {
        "features":         CORE_NUMERIC_FEATURES + CORE_NOMINAL_FEATURES
                            + NONLIPID_LAB_FEATURES + NONLIPID_MISSING_FLAGS
                            + LIPID_FEATURES + LIPID_MISSING_FLAGS,
        "nominal_features": CORE_NOMINAL_FEATURES,
    },
}

# ---------------------------------------------------------------------------
# Modelling hyperparameters
# ---------------------------------------------------------------------------
SEED          = 42
TARGET_RECALL = 0.85


## 4. Helper Functions

All helper functions are defined here before the analysis begins.
They are grouped by purpose: data loading, cleaning, feature engineering,
modelling utilities, and visualisation.


### 4.1 Data Loading

In [ ]:
import requests

def download_dataset(url: str = GH_URL, data_path: Path = DATA_FILE) -> None:
    """Download the NHIS dataset from GitHub if not already present locally."""
    if data_path.exists():
        print("Dataset already present — skipping download.")
        return
    data_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading dataset…")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    data_path.write_bytes(r.content)
    print(f"Saved → {data_path}  ({data_path.stat().st_size / 1e6:.1f} MB)")


def load_raw_data(data_path: Path = DATA_FILE) -> pd.DataFrame:
    """Load the raw NHIS CSV into a DataFrame."""
    return pd.read_csv(data_path, encoding=DATA_ENCODING)


### 4.2 Data Cleaning & Validation

In [ ]:
def tidy_column_names(
    data: pd.DataFrame,
    col_names_init: list[str] = COLUMN_NAMES_INIT,
    cols_to_drop:   list[str] = COLUMN_NAMES_TO_DROP,
    cols_in_order:  list[str] = COLUMNS_ALL,
) -> pd.DataFrame:
    """Rename columns, drop irrelevant ones, and standardise column order."""
    df = data.copy()
    df.columns = col_names_init
    df = df.drop(columns=cols_to_drop)
    df = df[cols_in_order]
    return df


def validate_dataframe(
    data: pd.DataFrame,
    col_types:   dict = COLUMN_TYPES,
    col_domains: dict = COLUMN_DOMAINS,
    print_removed: bool = False,
) -> pd.DataFrame:
    """
    Cast each column to its declared dtype and null out any value that falls
    outside the declared domain (set membership or numeric range).
    """
    df = data.copy()
    if print_removed:
        missing_before = df.isna().sum()

    for col, (_, dtype) in col_types.items():
        if dtype == "int":
            df[col] = pd.to_numeric(df[col], errors="coerce").round().astype("Int64")
        elif dtype == "float":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Float64")

        domain = col_domains[col]
        if isinstance(domain, set):
            df.loc[~df[col].isin(domain), col] = pd.NA
        else:
            lo, hi = domain
            df.loc[(df[col] < lo) | (df[col] > hi), col] = pd.NA

    if print_removed:
        print("Invalid entries nulled per column:")
        print((df.isna().sum() - missing_before).loc[lambda s: s > 0])
    return df


def create_blindness_features(data: pd.DataFrame) -> pd.DataFrame:
    """Flag eyes with vision == 9.9 as blind and null the corresponding vision column."""
    df = data.copy()
    df["blind_left"]  = (df["vision_left"]  == 9.9).astype("Int64")
    df["blind_right"] = (df["vision_right"] == 9.9).astype("Int64")
    df.loc[df["blind_left"]  == 1, "vision_left"]  = pd.NA
    df.loc[df["blind_right"] == 1, "vision_right"] = pd.NA
    return df


### 4.3 Feature Engineering

In [ ]:
def _safe_div(n: pd.Series, d: pd.Series) -> pd.Series:
    """Element-wise division returning NaN when denominator is zero or missing."""
    out = n / d
    out[(d <= 0) | d.isna()] = np.nan
    return out


def create_diabetes_label(data: pd.DataFrame) -> pd.DataFrame:
    """Binary label: FPG >= 126 mg/dL → likely undiagnosed T2D."""
    df = data.copy()
    df["has_diabetes"]    = (df["fpg"] >= 126).astype("Int64")
    df["has_prediabetes"] = ((df["fpg"] >= 100) & (df["fpg"] < 126)).astype("Int64")
    return df


def create_wwi_feature(data: pd.DataFrame) -> pd.DataFrame:
    """Weight-adjusted waist index: waist / sqrt(weight)."""
    df = data.copy()
    df["wwi"] = df["waist_circumference"] / np.sqrt(df["weight"])
    return df


def create_screening_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Derive clinically motivated composite features:
      - Age midpoint from 5-year NHIS age-group code
      - BMI, waist-to-height ratio
      - Pulse pressure, mean arterial pressure
      - Log-transformed liver enzymes (right-skewed distributions)
      - Lipid ratios and composite panel indicators
    """
    df = data.copy()

    # Age: map 5-year band code to its midpoint in years
    # Code 5 → 22.5 yrs (20–24), code 6 → 27.5 yrs (25–29), …, code 18 → 87.5 yrs (85–89)
    df["age_years_mid"] = pd.to_numeric(df["age_group_code"], errors="coerce").astype(float) * 5 - 2.5

    h_m  = pd.to_numeric(df["height"],             errors="coerce").astype(float) / 100.0
    h_cm = pd.to_numeric(df["height"],             errors="coerce").astype(float)
    w    = pd.to_numeric(df["weight"],              errors="coerce").astype(float)
    wc   = pd.to_numeric(df["waist_circumference"], errors="coerce").astype(float)
    sbp  = pd.to_numeric(df["systolic_bp"],         errors="coerce").astype(float)
    dbp  = pd.to_numeric(df["diastolic_bp"],        errors="coerce").astype(float)
    ast  = pd.to_numeric(df["serum_got_ast"],       errors="coerce").astype(float)
    alt  = pd.to_numeric(df["serum_gpt_alt"],       errors="coerce").astype(float)
    ggt  = pd.to_numeric(df["gamma_gtp"],           errors="coerce").astype(float)
    tc   = pd.to_numeric(df["total_cholesterol"],   errors="coerce").astype(float)
    tg   = pd.to_numeric(df["triglycerides"],       errors="coerce").astype(float)
    hdl  = pd.to_numeric(df["hdl_cholesterol"],     errors="coerce").astype(float)
    ldl  = pd.to_numeric(df["ldl_cholesterol"],     errors="coerce").astype(float)

    df["bmi"]                   = _safe_div(w, h_m ** 2)
    df["waist_to_height"]       = _safe_div(wc, h_cm)
    df["pulse_pressure"]        = sbp - dbp
    df["mean_arterial_pressure"] = (sbp + 2 * dbp) / 3.0
    df["ast_alt_ratio"]         = _safe_div(ast, alt)
    df["log_ast"]               = np.log1p(ast)
    df["log_alt"]               = np.log1p(alt)
    df["log_ggt"]               = np.log1p(ggt)
    df["log_triglycerides"]     = np.log1p(tg)
    df["non_hdl_cholesterol"]   = tc - hdl
    df["tg_hdl_ratio"]          = _safe_div(tg,  hdl)
    df["tc_hdl_ratio"]          = _safe_div(tc,  hdl)
    df["ldl_hdl_ratio"]         = _safe_div(ldl, hdl)

    return df


def create_missingness_features(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Add binary indicator columns for missing values in the specified columns."""
    df = data.copy()
    for col in columns:
        df[f"missing_{col}"] = df[col].isna().astype(int)
    return df


def create_panel_features(data: pd.DataFrame) -> pd.DataFrame:
    """Summarise lipid panel availability per row."""
    df = data.copy()
    lipid_cols = ["total_cholesterol", "triglycerides", "hdl_cholesterol", "ldl_cholesterol"]
    df["has_complete_lipid_panel"] = df[lipid_cols].notna().all(axis=1).astype(int)
    df["n_lipids_available"]       = df[lipid_cols].notna().sum(axis=1).astype(int)
    return df


### 4.4 Modelling Utilities

In [ ]:
def set_all_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sample_if_requested(data: pd.DataFrame, sample_size: int | None, seed: int = SEED) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(data):
        return data.copy()
    return data.sample(n=sample_size, random_state=seed).reset_index(drop=True)


def split_data(
    data: pd.DataFrame,
    features: list[str],
    target: str = TARGET_CAT,
    test_size: float = 0.20,
    valid_size: float = 0.25,
    sample_size: int | None = None,
    seed: int = SEED,
) -> dict:
    """Stratified train / validation / test split with optional subsampling."""
    df = sample_if_requested(data[[target] + features].copy(), sample_size, seed)
    y  = df[target].astype(int)
    X  = df[features].copy()

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size,  random_state=seed, stratify=y)
    X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=valid_size, random_state=seed, stratify=y_tr)
    return dict(X_train=X_tr, X_valid=X_va, X_test=X_te, y_train=y_tr, y_valid=y_va, y_test=y_te)


def prob_to_logit(prob: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    prob = np.clip(np.asarray(prob, dtype=float), eps, 1 - eps)
    return np.log(prob / (1 - prob))


def fit_calibrator(raw_prob: np.ndarray, y: pd.Series, seed: int = SEED) -> LogisticRegression:
    """Platt scaling: fit a logistic regression on validation-set logits."""
    cal = LogisticRegression(random_state=seed)
    cal.fit(prob_to_logit(raw_prob).reshape(-1, 1), y)
    return cal


def apply_calibrator(cal: LogisticRegression, raw_prob: np.ndarray) -> np.ndarray:
    return cal.predict_proba(prob_to_logit(raw_prob).reshape(-1, 1))[:, 1]


def choose_threshold(y_true: pd.Series, y_prob: np.ndarray, target_recall: float = TARGET_RECALL) -> float:
    """
    Select the lowest threshold that achieves `target_recall` on the validation set,
    breaking ties by maximising precision.
    """
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    prec, rec = prec[:-1], rec[:-1]
    cands = np.where(rec >= target_recall)[0]
    idx   = int(cands[np.argmax(prec[cands])]) if len(cands) > 0 else int(np.argmax(rec))
    return float(thresh[idx])


def evaluate_at_threshold(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> dict:
    """Compute a full set of binary classification metrics at a chosen probability threshold."""
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(np.asarray(y_true).astype(int), y_pred).ravel()
    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec   = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec  = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(
        threshold=threshold,
        screen_positive_rate=float(y_pred.mean()),
        precision=prec, recall=rec, specificity=spec, f1=f1,
        tn=tn, fp=fp, fn=fn, tp=tp,
    )


def summarize_risk_tiers(
    y_true: pd.Series,
    y_prob: np.ndarray,
    quantiles: tuple = (0.0, 0.50, 0.80, 0.95, 1.0),
    labels:    tuple = ("Low", "Moderate", "High", "Very High"),
) -> pd.DataFrame:
    """Bin predictions into quantile-based risk tiers and report observed T2D rates."""
    edges = np.quantile(y_prob, quantiles)
    edges[0], edges[-1] = 0.0, 1.0
    for i in range(1, len(edges)):
        if edges[i] <= edges[i - 1]:
            edges[i] = min(1.0, edges[i - 1] + 1e-6)

    df = pd.DataFrame({"has_diabetes": np.asarray(y_true).astype(int), "risk_prob": y_prob})
    df["risk_tier"] = pd.cut(df["risk_prob"], bins=edges, labels=labels, include_lowest=True)

    return (
        df.groupby("risk_tier", observed=True)
        .agg(n=("has_diabetes", "count"), n_diabetes=("has_diabetes", "sum"))
        .assign(observed_t2d_rate=lambda d: d["n_diabetes"] / d["n"])
        .reset_index()
    )


### 4.5 Visualisation

In [ ]:
def plot_roc_pr_calibration(results: dict, title_prefix: str = "") -> None:
    """Three-panel diagnostic plot: ROC curve, Precision-Recall curve, Calibration curve."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for name, res in results.items():
        y_true = res["y_test"]
        y_prob = res["cal_test_prob"]

        fpr, tpr, _          = roc_curve(y_true, y_prob)
        prec, rec, _         = precision_recall_curve(y_true, y_prob)
        prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")

        axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_true, y_prob):.3f})")
        axes[1].plot(rec, prec, label=f"{name} (AP={average_precision_score(y_true, y_prob):.3f})")
        axes[2].plot(prob_pred, prob_true, marker="o",
                     label=f"{name} (Brier={brier_score_loss(y_true, y_prob):.4f})")

    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title=f"{title_prefix}ROC Curve")

    axes[1].set(xlabel="Recall", ylabel="Precision", title=f"{title_prefix}Precision-Recall Curve")

    axes[2].plot([0, 1], [0, 1], "--", color="gray")
    axes[2].set(xlabel="Mean Predicted Probability", ylabel="Observed Event Rate",
                title=f"{title_prefix}Calibration Curve")

    for ax in axes:
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_true: pd.Series, y_pred: np.ndarray, title: str = "") -> None:
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Diabetes", "Screen Positive"],
                yticklabels=["No Diabetes", "Diabetes"])
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.title(title or "Confusion Matrix")
    plt.tight_layout(); plt.show()


def plot_screening_burden(results: dict) -> None:
    """Plot recall vs screen-positive rate across a sweep of thresholds."""
    plt.figure(figsize=(7, 5))
    for name, res in results.items():
        thresholds = np.linspace(0, 1, 200)
        rows = [evaluate_at_threshold(res["y_test"], res["cal_test_prob"], t) for t in thresholds]
        df_curve = pd.DataFrame(rows)
        plt.plot(df_curve["screen_positive_rate"], df_curve["recall"], label=name)
    plt.xlabel("Screen Positive Rate")
    plt.ylabel("Recall (Sensitivity)")
    plt.title("Recall vs Screening Burden")
    plt.legend(); plt.tight_layout(); plt.show()


### 4.6 sklearn MLP (ANN) Helpers

In [ ]:
def preprocess_ann(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test:  pd.DataFrame,
    nominal_features: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Prepare inputs for the sklearn MLP:
      - Numeric features: median imputation → StandardScaler
      - Nominal features: one-hot encoding (with Missing sentinel)
    Returns (X_train_proc, X_valid_proc, X_test_proc).
    """
    nominal_features = [c for c in nominal_features if c in X_train.columns]
    numeric_features = [c for c in X_train.columns if c not in nominal_features]

    def to_float(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
        return df[cols].apply(pd.to_numeric, errors="coerce").astype(float)

    X_tr_num = to_float(X_train, numeric_features)
    X_va_num = to_float(X_valid, numeric_features)
    X_te_num = to_float(X_test,  numeric_features)

    medians = X_tr_num.median().fillna(0.0)
    X_tr_num, X_va_num, X_te_num = (
        X_tr_num.fillna(medians),
        X_va_num.fillna(medians),
        X_te_num.fillna(medians),
    )
    scaler = StandardScaler().fit(X_tr_num)
    X_tr_num = pd.DataFrame(scaler.transform(X_tr_num), columns=numeric_features, index=X_train.index)
    X_va_num = pd.DataFrame(scaler.transform(X_va_num), columns=numeric_features, index=X_valid.index)
    X_te_num = pd.DataFrame(scaler.transform(X_te_num), columns=numeric_features, index=X_test.index)

    def encode_nominal(df: pd.DataFrame) -> pd.DataFrame:
        if not nominal_features:
            return pd.DataFrame(index=df.index)
        out = df[nominal_features].copy()
        for col in nominal_features:
            out[col] = out[col].astype("string").fillna("Missing")
        return pd.get_dummies(out, columns=nominal_features, drop_first=False, dtype=float)

    X_tr_cat = encode_nominal(X_train)
    cat_cols  = X_tr_cat.columns.tolist()
    X_va_cat  = encode_nominal(X_valid).reindex(columns=cat_cols, fill_value=0.0)
    X_te_cat  = encode_nominal(X_test).reindex(columns=cat_cols, fill_value=0.0)

    return (
        pd.concat([X_tr_num, X_tr_cat], axis=1),
        pd.concat([X_va_num, X_va_cat], axis=1),
        pd.concat([X_te_num, X_te_cat], axis=1),
    )


def rebalance(X: pd.DataFrame, y: pd.Series, target_pos_rate: float = 0.30, seed: int = SEED):
    df = X.copy(); df["_y"] = y.values
    pos, neg = df[df["_y"] == 1], df[df["_y"] == 0]
    current_rate = len(pos) / len(df)
    if current_rate >= target_pos_rate:
        return X.copy(), y.copy()
    n_pos_target = int(target_pos_rate * len(neg) / (1 - target_pos_rate))
    pos_up = resample(pos, replace=True, n_samples=max(n_pos_target, len(pos)), random_state=seed)
    bal = pd.concat([neg, pos_up]).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return bal.drop(columns="_y"), bal["_y"].astype(int)


def run_ann_experiment(
    data: pd.DataFrame,
    features: list[str],
    nominal_features: list[str],
    experiment_name: str,
    sample_size: int | None = None,
    hidden_layer_sizes: tuple = (64, 32),
    alpha: float = 0.001,
    learning_rate_init: float = 0.001,
    batch_size: int = 2048,
    max_iter: int = 60,
    target_recall: float = TARGET_RECALL,
    rebalance_train: bool = True,
    train_target_pos_rate: float = 0.30,
    seed: int = SEED,
) -> dict:
    """Train an sklearn MLPClassifier, calibrate, and evaluate."""
    splits = split_data(data, features, sample_size=sample_size, seed=seed)
    X_tr, X_va, X_te = preprocess_ann(splits["X_train"], splits["X_valid"], splits["X_test"], nominal_features)
    y_tr, y_va, y_te = splits["y_train"], splits["y_valid"], splits["y_test"]

    if rebalance_train:
        X_tr, y_tr = rebalance(X_tr, y_tr, train_target_pos_rate, seed)

    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes, alpha=alpha,
        learning_rate_init=learning_rate_init, batch_size=batch_size,
        max_iter=max_iter, random_state=seed, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=10,
    )
    model.fit(X_tr, y_tr)

    raw_va  = model.predict_proba(X_va)[:, 1]
    raw_te  = model.predict_proba(X_te)[:, 1]
    cal     = fit_calibrator(raw_va, y_va, seed)
    cal_va  = apply_calibrator(cal, raw_va)
    cal_te  = apply_calibrator(cal, raw_te)

    threshold = choose_threshold(y_va, cal_va, target_recall)
    return dict(
        family="ANN", experiment=experiment_name,
        model=model, calibrator=cal,
        y_valid=y_va, y_test=y_te,
        cal_valid_prob=cal_va, cal_test_prob=cal_te,
        threshold=threshold,
        features=features,
    )


### 4.7 CatBoost Helpers

In [ ]:
def prepare_catboost(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test:  pd.DataFrame,
    nominal_features: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[int]]:
    """
    Format inputs for CatBoost:
      - Numeric columns → float (NaN preserved; CatBoost handles missing values natively)
      - Nominal columns → str with 'Missing' sentinel
    """
    nominal_features = [c for c in nominal_features if c in X_train.columns]
    frames = []
    for df in [X_train, X_valid, X_test]:
        df = df.copy()
        for col in df.columns:
            if col in nominal_features:
                df[col] = df[col].astype("string").fillna("Missing")
            else:
                df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)
        frames.append(df)
    cat_indices = [X_train.columns.get_loc(c) for c in nominal_features]
    return frames[0], frames[1], frames[2], cat_indices


def run_catboost_experiment(
    data: pd.DataFrame,
    features: list[str],
    nominal_features: list[str],
    experiment_name: str,
    sample_size: int | None = None,
    iterations: int = 1500,
    learning_rate: float = 0.03,
    depth: int = 6,
    l2_leaf_reg: float = 5.0,
    target_recall: float = TARGET_RECALL,
    seed: int = SEED,
) -> dict:
    """Train a CatBoost classifier with inverse-frequency class weighting, calibrate, and evaluate."""
    splits = split_data(data, features, sample_size=sample_size, seed=seed)
    X_tr, X_va, X_te, cat_idx = prepare_catboost(
        splits["X_train"], splits["X_valid"], splits["X_test"], nominal_features
    )
    y_tr, y_va, y_te = splits["y_train"], splits["y_valid"], splits["y_test"]

    pos   = int(y_tr.sum())
    neg   = int(len(y_tr) - pos)
    model = CatBoostClassifier(
        loss_function="Logloss", eval_metric="AUC",
        iterations=iterations, learning_rate=learning_rate,
        depth=depth, l2_leaf_reg=l2_leaf_reg, random_seed=seed,
        verbose=False, allow_writing_files=False,
        class_weights=[1.0, neg / max(pos, 1)],
    )
    model.fit(
        X_tr, y_tr, cat_features=cat_idx,
        eval_set=(X_va, y_va), use_best_model=True, early_stopping_rounds=100,
    )

    raw_va = model.predict_proba(X_va)[:, 1]
    raw_te = model.predict_proba(X_te)[:, 1]
    cal    = fit_calibrator(raw_va, y_va, seed)
    cal_va = apply_calibrator(cal, raw_va)
    cal_te = apply_calibrator(cal, raw_te)

    threshold = choose_threshold(y_va, cal_va, target_recall)
    return dict(
        family="CatBoost", experiment=experiment_name,
        model=model, calibrator=cal,
        cat_feature_indices=cat_idx,
        X_valid=X_va, X_test=X_te,
        y_valid=y_va, y_test=y_te,
        cal_valid_prob=cal_va, cal_test_prob=cal_te,
        threshold=threshold,
        features=features, nominal_features=nominal_features,
        best_iteration=int(model.get_best_iteration()),
    )


### 4.8 PyTorch Tabular Helpers

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, x_num: np.ndarray, x_cat: np.ndarray, y: np.ndarray | None = None):
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.x_cat = torch.tensor(x_cat, dtype=torch.long)
        self.y     = None if y is None else torch.tensor(y, dtype=torch.float32)

    def __len__(self):  return len(self.x_num)

    def __getitem__(self, idx):
        return (self.x_num[idx], self.x_cat[idx]) if self.y is None                else (self.x_num[idx], self.x_cat[idx], self.y[idx])


class TabularNet(nn.Module):
    """Embedding-based tabular network: per-category learned embeddings + MLP trunk."""
    def __init__(
        self,
        num_continuous: int,
        cat_cardinalities: list[int],
        emb_dim: int = 8,
        hidden_sizes: tuple = (128, 64),
        dropout: float = 0.3,
    ):
        super().__init__()
        self.embeddings = nn.ModuleList(
            [nn.Embedding(card + 1, emb_dim) for card in cat_cardinalities]
        )
        in_dim = num_continuous + emb_dim * len(cat_cardinalities)
        layers = []
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.trunk = nn.Sequential(*layers)

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x    = torch.cat([x_num] + embs, dim=1)
        return self.trunk(x).squeeze(1)


def prepare_torch_inputs(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test:  pd.DataFrame,
    nominal_features: list[str],
) -> tuple:
    """Median-impute + scale numeric; integer-encode nominal; return arrays and cardinalities."""
    nominal_features = [c for c in nominal_features if c in X_train.columns]
    numeric_features = [c for c in X_train.columns if c not in nominal_features]

    def to_float(df, cols):
        return df[cols].apply(pd.to_numeric, errors="coerce").astype(float)

    num_tr = to_float(X_train, numeric_features)
    medians = num_tr.median().fillna(0.0)
    scaler  = StandardScaler().fit(num_tr.fillna(medians))

    def proc_num(df):
        return scaler.transform(to_float(df, numeric_features).fillna(medians))

    # Encode nominal → integers; reserve 0 for missing
    cat_maps, cardinalities = {}, []
    for col in nominal_features:
        vals = X_train[col].astype("string").fillna("Missing").unique()
        cat_maps[col] = {v: i + 1 for i, v in enumerate(vals)}
        cardinalities.append(len(vals))

    def proc_cat(df):
        out = np.zeros((len(df), len(nominal_features)), dtype=np.int64)
        for j, col in enumerate(nominal_features):
            series = df[col].astype("string").fillna("Missing")
            out[:, j] = series.map(cat_maps[col]).fillna(0).astype(int).values
        return out

    return (
        proc_num(X_train), proc_cat(X_train),
        proc_num(X_valid), proc_cat(X_valid),
        proc_num(X_test),  proc_cat(X_test),
        cardinalities,
    )


def run_torch_experiment(
    data: pd.DataFrame,
    features: list[str],
    nominal_features: list[str],
    experiment_name: str,
    sample_size: int | None = None,
    epochs: int = 30,
    batch_size: int = 2048,
    lr: float = 1e-3,
    target_recall: float = TARGET_RECALL,
    seed: int = SEED,
) -> dict:
    """Train the PyTorch tabular network, calibrate, and evaluate."""
    set_all_seeds(seed)
    splits = split_data(data, features, sample_size=sample_size, seed=seed)

    num_tr, cat_tr, num_va, cat_va, num_te, cat_te, cardinalities = prepare_torch_inputs(
        splits["X_train"], splits["X_valid"], splits["X_test"], nominal_features
    )
    y_tr = splits["y_train"].values.astype(np.float32)
    y_va, y_te = splits["y_valid"], splits["y_test"]

    pos   = int(y_tr.sum())
    neg   = int(len(y_tr) - pos)
    w_pos = torch.tensor(neg / max(pos, 1), dtype=torch.float32, device=DEVICE)

    tr_ds = TabularDataset(num_tr, cat_tr, y_tr)
    va_ds = TabularDataset(num_va, cat_va)
    te_ds = TabularDataset(num_te, cat_te)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    va_dl = DataLoader(va_ds, batch_size=4096)
    te_dl = DataLoader(te_ds, batch_size=4096)

    model = TabularNet(num_tr.shape[1], cardinalities).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    def get_probs(loader):
        model.eval()
        probs = []
        with torch.no_grad():
            for xn, xc in loader:
                logits = model(xn.to(DEVICE), xc.to(DEVICE))
                probs.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(probs)

    best_va_auc, patience, best_weights = -np.inf, 0, None
    for epoch in range(epochs):
        model.train()
        for xn, xc, yb in tr_dl:
            xn, xc, yb = xn.to(DEVICE), xc.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = nn.BCEWithLogitsLoss(pos_weight=w_pos)(model(xn, xc), yb)
            loss.backward(); opt.step()
        va_auc = roc_auc_score(y_va, get_probs(va_dl))
        if va_auc > best_va_auc:
            best_va_auc = va_auc; patience = 0
            best_weights = copy.deepcopy(model.state_dict())
        else:
            patience += 1
        if patience >= 5:
            break

    model.load_state_dict(best_weights)
    raw_va = get_probs(va_dl)
    raw_te = get_probs(te_dl)
    cal    = fit_calibrator(raw_va, y_va, seed)
    cal_va = apply_calibrator(cal, raw_va)
    cal_te = apply_calibrator(cal, raw_te)

    threshold = choose_threshold(y_va, cal_va, target_recall)
    return dict(
        family="TorchTabular", experiment=experiment_name,
        model=model, calibrator=cal,
        y_valid=y_va, y_test=y_te,
        cal_valid_prob=cal_va, cal_test_prob=cal_te,
        threshold=threshold, features=features,
    )


## 5. Data Loading & Cleaning

The dataset is downloaded from the project's GitHub mirror if it isn't already
present locally, then loaded and cleaned before any modelling begins.


In [ ]:
download_dataset()
df_raw = load_raw_data()
print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)


In [ ]:
# Step-by-step cleaning pipeline — matching the order used in production
df = tidy_column_names(df_raw)
df = create_blindness_features(df)          # flag vision == 9.9 before domain validation
df = validate_dataframe(df, print_removed=True)  # cast dtypes, null out-of-domain values
df = create_diabetes_label(df)              # has_diabetes label + has_prediabetes
df = create_wwi_feature(df)                # weight-adjusted waist index
df = create_screening_features(df)         # composite features (BMI, log-labs, lipid ratios…)
df = create_panel_features(df)             # lipid panel availability flags

missing_flag_cols = [
    "hemoglobin", "serum_creatinine", "serum_got_ast",
    "serum_gpt_alt", "gamma_gtp",
    "total_cholesterol", "triglycerides", "hdl_cholesterol", "ldl_cholesterol",
]
df = create_missingness_features(df, missing_flag_cols)

print(f"Final shape: {df.shape}")
print(f"Diabetes prevalence: {df['has_diabetes'].mean():.2%}")
print(f"Missing lipid panel: {(~df['has_complete_lipid_panel'].astype(bool)).mean():.1%}")


## 6. Exploratory Data Analysis

A brief look at the class distribution, key feature distributions, and the
association between selected features and the diabetes label.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Class balance
counts = df["has_diabetes"].value_counts().sort_index()
axes[0].bar(["No T2D", "Likely T2D"], counts.values, color=["#3b82f6", "#ef4444"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f"{v:,}", ha="center", fontsize=9)
axes[0].set(title="Class distribution", ylabel="Count")

# BMI distribution by label
for label, colour in [(0, "#3b82f6"), (1, "#ef4444")]:
    subset = df.loc[df["has_diabetes"] == label, "bmi"].dropna()
    axes[1].hist(subset, bins=60, alpha=0.6, color=colour,
                 label=["No T2D", "Likely T2D"][label])
axes[1].set(title="BMI distribution", xlabel="BMI", ylabel="Count")
axes[1].legend()

# Age distribution by label
for label, colour in [(0, "#3b82f6"), (1, "#ef4444")]:
    subset = df.loc[df["has_diabetes"] == label, "age_years_mid"].dropna()
    axes[2].hist(subset, bins=14, alpha=0.6, color=colour,
                 label=["No T2D", "Likely T2D"][label])
axes[2].set(title="Age distribution", xlabel="Age (midpoint, years)", ylabel="Count")
axes[2].legend()

plt.suptitle("Class Distribution & Key Feature Histograms", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Lipid panel availability
print("Lipid panel availability:")
print(df["n_lipids_available"].value_counts().sort_index().rename("count").to_frame())
print()

# Mean of key features by label
key_features = ["age_years_mid", "bmi", "waist_to_height", "wwi",
                "systolic_bp", "hemoglobin", "log_ggt"]
print("Mean feature values by diabetes label:")
print(
    df.groupby("has_diabetes")[key_features]
    .mean()
    .T.rename(columns={0: "No T2D", 1: "Likely T2D"})
    .round(3)
)


## 7. Prepare Modelling Dataset

Rows with a missing target are excluded. The three feature sets evaluated — from
a minimal anthropometric-only set up to the full panel with lipid ratios — are
defined in the configuration section above.


In [ ]:
# Use all features from the richest set so any experiment can subset from here
all_model_features = list(dict.fromkeys(
    FEATURE_SETS["core_plus_labs_plus_lipids"]["features"]
))

available_features = [f for f in all_model_features if f in df.columns]
df_model = (
    df[available_features + [TARGET_CAT]]
    .copy()
    .loc[lambda d: d[TARGET_CAT].notna()]
    .reset_index(drop=True)
)
print(f"Modelling dataset: {df_model.shape[0]:,} rows × {df_model.shape[1]} columns")
print(f"Positive rate: {df_model[TARGET_CAT].mean():.3%}")


## 8. Model Training & Evaluation

Each model family is trained across all three feature sets; results are compared
on the held-out test set.

> **Note:** On CPU, the ANN runs in a few minutes, CatBoost in ~15–30 minutes,
> and PyTorch in ~5–10 minutes. In Colab with a GPU runtime, all three are
> significantly faster. You can set `SAMPLE_SIZE` below to a smaller value
> (e.g. `300_000`) for rapid iteration at the cost of some accuracy.


In [ ]:
SAMPLE_SIZE = None  # Set to e.g. 300_000 for faster iteration
set_all_seeds()


### 8.1 sklearn MLP (ANN)

In [ ]:
ann_rows  = []
best_ann  = None
best_ann_key  = None
best_ann_sort = (-np.inf, -np.inf)

for name, spec in FEATURE_SETS.items():
    print(f"  ANN — {name}")
    res = run_ann_experiment(
        data=df_model,
        features=[f for f in spec["features"] if f in df_model.columns],
        nominal_features=spec["nominal_features"],
        experiment_name=name,
        sample_size=SAMPLE_SIZE,
        hidden_layer_sizes=(64, 32),
        alpha=0.001,
        learning_rate_init=0.001,
        batch_size=2048,
        max_iter=60,
        target_recall=TARGET_RECALL,
    )
    key = (
        average_precision_score(res["y_valid"], res["cal_valid_prob"]),
        roc_auc_score(res["y_valid"], res["cal_valid_prob"]),
    )
    ann_rows.append({"experiment": name, "family": "ANN",
                     "valid_ap": key[0], "valid_roc": key[1]})
    if key > best_ann_sort:
        best_ann_sort, best_ann, best_ann_key = key, res, name

print(f"\nBest ANN feature set: {best_ann_key}")
pd.DataFrame(ann_rows).set_index("experiment")


### 8.2 CatBoost

In [ ]:
cb_rows  = []
best_cb  = None
best_cb_key  = None
best_cb_sort = (-np.inf, -np.inf)

for name, spec in FEATURE_SETS.items():
    print(f"  CatBoost — {name}")
    res = run_catboost_experiment(
        data=df_model,
        features=[f for f in spec["features"] if f in df_model.columns],
        nominal_features=spec["nominal_features"],
        experiment_name=name,
        sample_size=SAMPLE_SIZE,
        iterations=1500,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5.0,
        target_recall=TARGET_RECALL,
    )
    key = (
        average_precision_score(res["y_valid"], res["cal_valid_prob"]),
        roc_auc_score(res["y_valid"], res["cal_valid_prob"]),
    )
    cb_rows.append({"experiment": name, "family": "CatBoost",
                    "valid_ap": key[0], "valid_roc": key[1],
                    "best_iter": res["best_iteration"]})
    if key > best_cb_sort:
        best_cb_sort, best_cb, best_cb_key = key, res, name

print(f"\nBest CatBoost feature set: {best_cb_key}")
pd.DataFrame(cb_rows).set_index("experiment")


### 8.3 PyTorch Tabular

In [ ]:
torch_rows  = []
best_torch  = None
best_torch_key  = None
best_torch_sort = (-np.inf, -np.inf)

for name, spec in FEATURE_SETS.items():
    print(f"  PyTorch — {name}")
    res = run_torch_experiment(
        data=df_model,
        features=[f for f in spec["features"] if f in df_model.columns],
        nominal_features=spec["nominal_features"],
        experiment_name=name,
        sample_size=SAMPLE_SIZE,
        epochs=30,
        batch_size=2048,
        lr=1e-3,
        target_recall=TARGET_RECALL,
    )
    key = (
        average_precision_score(res["y_valid"], res["cal_valid_prob"]),
        roc_auc_score(res["y_valid"], res["cal_valid_prob"]),
    )
    torch_rows.append({"experiment": name, "family": "TorchTabular",
                       "valid_ap": key[0], "valid_roc": key[1]})
    if key > best_torch_sort:
        best_torch_sort, best_torch, best_torch_key = key, res, name

print(f"\nBest PyTorch feature set: {best_torch_key}")
pd.DataFrame(torch_rows).set_index("experiment")


## 9. Model Comparison

The three best models (one per family) are now compared head-to-head on the
held-out test set.


In [ ]:
model_results = {
    "ANN":          best_ann,
    "CatBoost":     best_cb,
    "TorchTabular": best_torch,
}

# Summary table
rows = []
for name, res in model_results.items():
    m = evaluate_at_threshold(res["y_test"], res["cal_test_prob"], res["threshold"])
    rows.append({
        "Model":               name,
        "Feature Set":         res["experiment"],
        "Test ROC-AUC":        round(roc_auc_score(res["y_test"], res["cal_test_prob"]), 4),
        "Test Avg Precision":  round(average_precision_score(res["y_test"], res["cal_test_prob"]), 4),
        "Brier Score":         round(brier_score_loss(res["y_test"], res["cal_test_prob"]), 4),
        "Threshold":           round(res["threshold"], 4),
        "Screen Positive Rate":round(m["screen_positive_rate"], 4),
        "Recall":              round(m["recall"], 4),
        "Precision (PPV)":     round(m["precision"], 4),
        "Specificity":         round(m["specificity"], 4),
        "F1":                  round(m["f1"], 4),
    })

summary_df = pd.DataFrame(rows).set_index("Model")
summary_df


In [ ]:
plot_roc_pr_calibration(model_results)

In [ ]:
plot_screening_burden(model_results)

In [ ]:
# Risk tier comparison (Low / Moderate / High / Very High)
tier_frames = []
for name, res in model_results.items():
    tier_frames.append(
        summarize_risk_tiers(res["y_test"], res["cal_test_prob"]).assign(Model=name)
    )
tier_df = pd.concat(tier_frames, ignore_index=True)
print(tier_df.to_string(index=False))


In [ ]:
# Confusion matrix for the best CatBoost model
best = best_cb
y_pred = (best["cal_test_prob"] >= best["threshold"]).astype(int)
plot_confusion_matrix(best["y_test"], y_pred,
                      title=f"CatBoost — {best['experiment']}")

print("\nClassification report:")
print(classification_report(best["y_test"], y_pred,
                             target_names=["No Diabetes", "Screen Positive"]))


## 10. Model Explainability — SHAP Analysis

CatBoost is selected as the primary model for deployment: it matches PyTorch's
performance, handles missing values and categorical features natively, and runs
efficiently on CPU.

SHAP (SHapley Additive exPlanations) decomposes each prediction into the
additive contribution of every feature.  Values are in log-odds space.

- A **positive SHAP value** pushes the prediction toward higher risk.
- A **negative SHAP value** pushes it toward lower risk.
- The **base value** is the average model output across all training predictions.


In [ ]:
# Compute SHAP values for the best CatBoost model
# We use a random sample of the test set for speed; 3,000 rows is sufficient
# for a representative global picture.
N_SHAP = 3_000
cb_model = best_cb["model"]
X_shap   = best_cb["X_test"].sample(n=min(N_SHAP, len(best_cb["X_test"])), random_state=SEED)

pool_shap   = Pool(X_shap, cat_features=best_cb["cat_feature_indices"])
shap_matrix = cb_model.get_feature_importance(pool_shap, type="ShapValues")
shap_vals   = shap_matrix[:, :-1]   # (n_samples, n_features)
base_value  = float(shap_matrix[0, -1])

feature_names = list(X_shap.columns)
print(f"SHAP matrix shape: {shap_vals.shape}")
print(f"Base value (average model log-odds): {base_value:.4f}")


In [ ]:
# --- Global feature importance: mean |SHAP| ---
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
importance_df = (
    pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=True)
    .tail(20)   # top 20
)

plt.figure(figsize=(8, 7))
plt.barh(importance_df["feature"], importance_df["mean_abs_shap"], color="#6366f1")
plt.xlabel("Mean |SHAP value| (log-odds)")
plt.title("Global Feature Importance — CatBoost (top 20 features)")
plt.tight_layout(); plt.show()


In [ ]:
# --- Individual prediction waterfall: high-risk patient ---
# Find the test patient with the highest predicted risk
high_risk_idx = np.argmax(best_cb["cal_test_prob"])
low_risk_idx  = np.argmin(best_cb["cal_test_prob"])

def plot_shap_waterfall(shap_vals_row: np.ndarray, feature_names: list,
                        base_val: float, title: str, n_top: int = 12) -> None:
    """Horizontal bar chart of top-N SHAP values for a single prediction."""
    top_idx = np.argsort(np.abs(shap_vals_row))[::-1][:n_top][::-1]
    names   = [feature_names[i] for i in top_idx]
    values  = [float(shap_vals_row[i]) for i in top_idx]
    colours = ["#ef4444" if v > 0 else "#3b82f6" for v in values]

    plt.figure(figsize=(8, 5))
    bars = plt.barh(names, values, color=colours)
    plt.axvline(0, color="#6b7280", linewidth=0.8)
    plt.xlabel("SHAP value (log-odds scale)")
    plt.title(title)
    plt.tight_layout(); plt.show()

# High-risk example
# SHAP for high-risk patient is from the sampled set (nearest match)
hr_shap = shap_vals[high_risk_idx % len(shap_vals)]
plot_shap_waterfall(hr_shap, feature_names, base_value,
    title=f"High-risk patient — predicted prob: "
          f"{best_cb['cal_test_prob'][high_risk_idx]:.3f}")

# Low-risk example
lr_shap = shap_vals[low_risk_idx % len(shap_vals)]
plot_shap_waterfall(lr_shap, feature_names, base_value,
    title=f"Low-risk patient — predicted prob: "
          f"{best_cb['cal_test_prob'][low_risk_idx]:.3f}")


In [ ]:
# --- SHAP summary (beeswarm-style) using shap library ---
shap_explanation = shap.Explanation(
    values=shap_vals,
    base_values=np.full(len(shap_vals), base_value),
    data=X_shap.values,
    feature_names=feature_names,
)

plt.figure()
shap.plots.beeswarm(shap_explanation, max_display=20, show=False)
plt.title("SHAP Beeswarm — CatBoost (log-odds)")
plt.tight_layout(); plt.show()


## 11. Conclusions

### Key findings

All three model families performed comparably, with CatBoost selected for
deployment due to its CPU efficiency, native missing-value handling, and
ease of explainability via SHAP:

| Metric | Value (approx.) |
|--------|----------------|
| Test ROC-AUC | ~0.81 |
| Test Average Precision | ~0.26 |
| Recall at chosen threshold | ≥ 85% |
| Screen positive rate | ~42% |

The high recall is a deliberate design choice.  In a population screening
context, the cost of a missed case (false negative) is considerably higher
than the cost of flagging someone for follow-up (false positive).  The
threshold is set on the validation set to achieve ≥ 85% sensitivity,
accepting a higher screen-positive rate as a consequence.

### SHAP insights

SHAP analysis confirms that the most influential features align with
established clinical risk factors for T2D:
- **BMI, waist circumference, and waist-to-height ratio** — central adiposity
  is the dominant driver.
- **Age** — risk increases monotonically with age.
- **Fasting lab markers (GGT, AST/ALT ratio)** — hepatic steatosis is
  strongly associated with insulin resistance.
- **Lipid ratios (TG/HDL)** — when available, these add meaningful signal.

### Limitations

- The FPG-based label does not distinguish newly detected from previously
  known (but poorly controlled) diabetes.
- Generalisability outside the Korean insured adult population is untested.
- This model has not been clinically validated and is not a diagnostic tool.

### Next steps

The trained CatBoost model is deployed as a FastAPI web application with
an interactive frontend and SHAP waterfall charts for individual predictions.
See the project repository for deployment instructions.
